# Day 5 — GBDT を建てる（勾配ブースティング）

**今日のゴール**
1. 森（道A＝独立に多数決）の先へ ＝ **道B＝前の木の"間違い"を順に直す（GBDT）**
2. 今日つかんだチューニングの勘どころ **「学習率(歩幅)は小さめ ＋ 木は多め」** を、自分の手で効かせる
3. 複雑データで、**森(0.795)を GBDT で超えられるか**に挑む

🧱 = 足場（私が用意した）　🎯 = 本筋（あなたが書く）　✋ = 走らせる前に予想する

> 復習: GBDT は森と違い「木を足しすぎると過学習する（test が山なりに下がる）」。
> だから **学習率を小さく＋木を多め** が安全策。ただし歩幅を小さくしたら本数も増やさないと"練習不足"で負ける。

## 🧱 1. 道具を読み込む（import）

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
# GradientBoostingClassifier は本筋なので、ここでは import しない（あなたが下で書く）

print('準備OK')

## 🧱 2. 複雑データを読み込む（交互作用＋ノイズ列あり・2000件15列）

Day 4 で森が圧勝した、あのデータ。森が得意＝GBDT も本領を出せる土俵。

In [ ]:
df = pd.read_csv('../../data/chat_sessions_complex.csv')

X = df.drop(columns='deepened')   # 手がかり（15列：本物の手がかり＋囮ノイズ列）
y = df['deepened']                # 本当の答え

print('データ:', df.shape)
df.head()

## 🧱 3. 比較相手：森（前回の王者）

複雑データでの森の成績を、交差検証で出しておく（≒0.795 が出るはず）。

In [ ]:
forest_cv = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=42), X, y, cv=10)
print(f'森(300本) : {forest_cv.mean():.3f} +/- {forest_cv.std():.3f}   <- これを GBDT で超えたい')

## ✋ 走らせる前に「予想」する

- チューニングした GBDT は、森の **≒0.795** を **超える？／互角？／届かない？**
- 学習率(歩幅)を小さくしたのに木の本数が少なすぎると、何が起きる？（"練習不足"）

予想をメモしてから、下を書いて答え合わせ。

## 🎯 4. ここが今日の本筋 — GBDT を建てて測る

ヒントだけ。完成形は渡しません。

**やること:**
1. **import**：GBDT も **森と同じ `sklearn.ensemble` の引き出し**にいる。クラス名は `GradientBoostingClassifier`。
2. **箱を作る**：今日のレシピを効かせる
   - `learning_rate=` … 小さめの歩幅（まず **0.1** あたり）
   - `n_estimators=` … 木は多め（まず **300** あたり）。※歩幅を小さくするほど本数は増やす
   - `random_state=42`（深さは指定しなくてOK）
3. **測る**：森と**同じ呼び方**で `cross_val_score(箱, X, y, cv=10)` を `gb_cv` に入れる
4. `gb_cv.mean()` と `.std()` を print

In [ ]:
# 🎯 ここを自分で埋める（... のままだと実行時にエラーになります）

# 1. import（GBDT も sklearn.ensemble。クラス名は GradientBoostingClassifier）
# from sklearn.ensemble import ...

# 2. GBDT の箱（learning_rate=小さめ, n_estimators=多め, random_state=42）
gb = ...

# 3. 測る（森と同じ cross_val_score の呼び方で、複雑データ X, y を）
gb_cv = ...

# 4. 結果を表示
# print(f'GBDT : {gb_cv.mean():.3f} +/- {gb_cv.std():.3f}')

## 🧱 5. 答え合わせ — 森 vs GBDT

`gb_cv` まで出せたら、このセルを実行。

In [ ]:
labels = ['forest(300)', 'GBDT (yours)']
means  = [forest_cv.mean(), gb_cv.mean()]
stds   = [forest_cv.std(),  gb_cv.std()]

fig, ax = plt.subplots(figsize=(5.5, 4))
bars = ax.bar(labels, means, yerr=stds, capsize=7, color=['#e0a458', '#3a6ea5'], edgecolor='#333')
ax.set_ylim(0.7, 0.86)
ax.set_ylabel('cross-val accuracy (10-fold)')
ax.set_title('Random forest vs GBDT (complex data)')
for b, m in zip(bars, means):
    ax.text(b.get_x() + b.get_width()/2, m + 0.004, f'{m:.3f}', ha='center')
fig.savefig('day5_gbdt_vs_forest.png', dpi=130, bbox_inches='tight')
plt.show()
print('森' if forest_cv.mean() > gb_cv.mean() else 'GBDT', 'の勝ち')